In [1]:
import pandas as pd

# Upload the dataset
from google.colab import files

uploaded = files.upload()

Saving fleet_fuel_dataset_10000.csv to fleet_fuel_dataset_10000.csv


In [2]:
import pandas as pd

df = pd.read_csv("fleet_fuel_dataset_10000.csv")

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (10000, 6)


,Distance_km,Vehicle_Type,Passengers,Nominal_L_per_100km,Duration_min,Fuel_Consumed_L
0,13.80,Sedan,1,4.71,25.23,0.76
1,16.39,Bus,22,14.62,33.42,3.57
2,28.71,Van,9,11.70,57.49,3.94
3,23.66,SUV,6,7.24,26.61,1.71
4,33.78,Sedan,4,7.84,60.05,3.39


In [4]:
print("Column Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

Column Names:
['Distance_km', 'Vehicle_Type', 'Passengers', 'Nominal_L_per_100km', 'Duration_min', 'Fuel_Consumed_L']

Data Types:
Distance_km            float64
Vehicle_Type            object
Passengers               int64
Nominal_L_per_100km    float64
Duration_min           float64
Fuel_Consumed_L        float64
dtype: object

Missing Values:
Distance_km            0
Vehicle_Type           0
Passengers             0
Nominal_L_per_100km    0
Duration_min           0
Fuel_Consumed_L        0
dtype: int64

Duplicate Rows:
0


In [5]:
# Remove duplicate rows
df = df.drop_duplicates()

# Remove rows with missing values
df = df.dropna()

# Reset index
df = df.reset_index(drop=True)

print("Shape after cleaning:", df.shape)

Shape after cleaning: (10000, 6)


In [6]:
output_file = "fleet_fuel_dataset_10000_preprocessed.csv"

df.to_csv(output_file, index=False)

print("Preprocessed dataset saved successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))

Preprocessed dataset saved successfully!
Rows: 10000
Columns: 6


In [7]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"
)

vehicle_encoded = encoder.fit_transform(
    df[["Vehicle_Type"]]
)

encoded_columns = encoder.get_feature_names_out(
    ["Vehicle_Type"]
)

vehicle_encoded_df = pd.DataFrame(
    vehicle_encoded,
    columns=encoded_columns,
    index=df.index
)

df_encoded = pd.concat(
    [
        df.drop(columns=["Vehicle_Type"]),
        vehicle_encoded_df
    ],
    axis=1
)

print("New Shape:", df_encoded.shape)
df_encoded.head()

New Shape: (10000, 9)


,Distance_km,Passengers,Nominal_L_per_100km,Duration_min,Fuel_Consumed_L,Vehicle_Type_Bus,Vehicle_Type_SUV,Vehicle_Type_Sedan,Vehicle_Type_Van
0,13.80,1,4.71,25.23,0.76,0.0,0.0,1.0,0.0
1,16.39,22,14.62,33.42,3.57,1.0,0.0,0.0,0.0
2,28.71,9,11.70,57.49,3.94,0.0,0.0,0.0,1.0
3,23.66,6,7.24,26.61,1.71,0.0,1.0,0.0,0.0
4,33.78,4,7.84,60.05,3.39,0.0,0.0,1.0,0.0


In [8]:
from sklearn.preprocessing import StandardScaler

numeric_columns = [
    "Distance_km",
    "Passengers",
    "Nominal_L_per_100km",
    "Duration_min",
    "Fuel_Consumed_L"
]

scaler = StandardScaler()

df_encoded[numeric_columns] = scaler.fit_transform(
    df_encoded[numeric_columns]
)

print("Scaling completed successfully!")
df_encoded.head()

Scaling completed successfully!


,Distance_km,Passengers,Nominal_L_per_100km,Duration_min,Fuel_Consumed_L,Vehicle_Type_Bus,Vehicle_Type_SUV,Vehicle_Type_Sedan,Vehicle_Type_Van
0,-0.700091,-0.678623,-1.195768,-0.696890,-0.792075,0.0,0.0,1.0,0.0
1,-0.537286,1.634603,1.181017,-0.398863,0.266069,1.0,0.0,0.0,0.0
2,0.237140,0.202606,0.480693,0.477025,0.405397,0.0,0.0,0.0,1.0
3,-0.080299,-0.127855,-0.588980,-0.646673,-0.434339,0.0,1.0,0.0,0.0
4,0.555837,-0.348163,-0.445078,0.570182,0.198287,0.0,0.0,1.0,0.0


In [9]:
# Features
X = df_encoded.drop(columns=["Fuel_Consumed_L"])

# Target
y = df_encoded["Fuel_Consumed_L"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("\nFeature columns:")
print(X.columns.tolist())

Features shape: (10000, 8)
Target shape: (10000,)

Feature columns:
['Distance_km', 'Passengers', 'Nominal_L_per_100km', 'Duration_min', 'Vehicle_Type_Bus', 'Vehicle_Type_SUV', 'Vehicle_Type_Sedan', 'Vehicle_Type_Van']


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training Features:", X_train.shape)
print("Testing Features:", X_test.shape)

print("Training Target:", y_train.shape)
print("Testing Target:", y_test.shape)

Training Features: (8000, 8)
Testing Features: (2000, 8)
Training Target: (8000,)
Testing Target: (2000,)


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Start from the cleaned original dataset
X_raw = df.drop(columns=["Fuel_Consumed_L", "Vehicle_Type"])
y = df["Fuel_Consumed_L"]

# Encode Vehicle Type
vehicle_encoded = pd.get_dummies(
    df["Vehicle_Type"],
    prefix="Vehicle_Type",
    dtype=int
)

X_raw = pd.concat([X_raw, vehicle_encoded], axis=1)

# Split BEFORE scaling
X_train, X_test, y_train, y_test = train_test_split(
    X_raw,
    y,
    test_size=0.20,
    random_state=42
)

# Scale numeric features using TRAINING data only
numeric_columns = [
    "Distance_km",
    "Passengers",
    "Nominal_L_per_100km",
    "Duration_min"
]

scaler = StandardScaler()

X_train[numeric_columns] = scaler.fit_transform(
    X_train[numeric_columns]
)

X_test[numeric_columns] = scaler.transform(
    X_test[numeric_columns]
)

print("Correct preprocessing completed!")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

Correct preprocessing completed!
X_train: (8000, 8)
X_test: (2000, 8)
y_train: (8000,)
y_test: (2000,)


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Separate features and target
X_raw = df.drop(columns=["Fuel_Consumed_L", "Vehicle_Type"])
y = df["Fuel_Consumed_L"]

# Encode Vehicle Type
vehicle_encoded = pd.get_dummies(
    df["Vehicle_Type"],
    prefix="Vehicle_Type",
    dtype=int
)

X_raw = pd.concat([X_raw, vehicle_encoded], axis=1)

# Split BEFORE scaling
X_train, X_test, y_train, y_test = train_test_split(
    X_raw,
    y,
    test_size=0.20,
    random_state=42
)

# Scale numerical features using TRAINING data only
numeric_columns = [
    "Distance_km",
    "Passengers",
    "Nominal_L_per_100km",
    "Duration_min"
]

scaler = StandardScaler()

X_train[numeric_columns] = scaler.fit_transform(
    X_train[numeric_columns]
)

X_test[numeric_columns] = scaler.transform(
    X_test[numeric_columns]
)

print("Correct preprocessing completed!")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

Correct preprocessing completed!
X_train: (8000, 8)
X_test: (2000, 8)
y_train: (8000,)
y_test: (2000,)


In [13]:
# Save preprocessed datasets

X_train.to_csv("X_train.csv", index=False)
X_test.to_csv("X_test.csv", index=False)

y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

print("Files saved successfully!")
print()
print("X_train.csv:", X_train.shape)
print("X_test.csv:", X_test.shape)
print("y_train.csv:", y_train.shape)
print("y_test.csv:", y_test.shape)

Files saved successfully!

X_train.csv: (8000, 8)
X_test.csv: (2000, 8)
y_train.csv: (8000,)
y_test.csv: (2000,)


In [14]:
import joblib

joblib.dump(scaler, "scaler.pkl")

print("Scaler saved successfully!")
print("File: scaler.pkl")

Scaler saved successfully!
File: scaler.pkl


In [15]:
import os

files = [
    "X_train.csv",
    "X_test.csv",
    "y_train.csv",
    "y_test.csv",
    "scaler.pkl"
]

print("Final Preprocessing Files:")
print("-" * 35)

for file in files:
    if os.path.exists(file):
        print("✅", file)
    else:
        print("❌", file)

Final Preprocessing Files:
-----------------------------------
✅ X_train.csv
✅ X_test.csv
✅ y_train.csv
✅ y_test.csv
✅ scaler.pkl


In [16]:
print("X_train sample:")
display(X_train.head())

print("\nX_test sample:")
display(X_test.head())

print("\ny_train sample:")
display(y_train.head())

print("\ny_test sample:")
display(y_test.head())

print("\nFinal Shapes:")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train sample:


,Distance_km,Passengers,Nominal_L_per_100km,Duration_min,Vehicle_Type_Bus,Vehicle_Type_SUV,Vehicle_Type_Sedan,Vehicle_Type_Van
9254,-0.221325,-0.345837,0.744194,0.172476,0,0,0,1
1561,-0.565587,-0.456200,-0.775874,-0.507544,0,0,1,0
1670,-0.551005,-0.566563,-0.871627,-0.412116,0,0,1,0
6087,-1.136186,-0.345837,-0.445529,-0.966476,0,0,1,0
6669,-0.288529,-0.345837,-1.312088,-0.713699,0,0,1,0



X_test sample:


,Distance_km,Passengers,Nominal_L_per_100km,Duration_min,Vehicle_Type_Bus,Vehicle_Type_SUV,Vehicle_Type_Sedan,Vehicle_Type_Van
6252,-0.614405,-0.566563,-0.428772,-0.723898,0,0,1,0
4684,-0.096428,0.537069,0.011689,-0.349468,0,0,0,1
1731,-0.908580,-0.566563,-0.201360,-0.779261,0,1,0,0
4742,-0.785584,-0.456200,-0.830932,-0.762506,0,0,1,0
4521,-0.348759,-0.566563,-0.718423,-0.375328,0,1,0,0



y_train sample:


,Fuel_Consumed_L
9254,3.39
1561,1.12
1670,0.98
6087,0.63
6669,0.88



y_test sample:


,Fuel_Consumed_L
6252,1.24
4684,3.10
1731,1.16
4742,0.91
4521,1.41



Final Shapes:
X_train: (8000, 8)
X_test: (2000, 8)
y_train: (8000,)
y_test: (2000,)


In [17]:
import zipfile
import os

files_to_zip = [
    "fleet_fuel_dataset_10000.csv",
    "X_train.csv",
    "X_test.csv",
    "y_train.csv",
    "y_test.csv",
    "scaler.pkl"
]

zip_name = "Fleet_Fuel_Dataset_Preprocessing.zip"

with zipfile.ZipFile(zip_name, "w") as zipf:
    for file in files_to_zip:
        if os.path.exists(file):
            zipf.write(file)

print("ZIP file created successfully!")
print("File:", zip_name)

ZIP file created successfully!
File: Fleet_Fuel_Dataset_Preprocessing.zip


In [18]:
from google.colab import files

files.download("Fleet_Fuel_Dataset_Preprocessing.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>